# TP 1 — Partie 5 : Iceberg ou Delta Lake ?

Toutes les tables du cours sont au format **Apache Iceberg**. Ce n'est pas le seul format de table ouvert : **Delta Lake**, créé par Databricks, joue exactement le même rôle. Les deux reposent sur des fichiers **Parquet**, auxquels ils ajoutent des **métadonnées** : le schéma, la liste des fichiers qui composent la table, l'historique des versions.

Dans cette partie courte (30 à 45 minutes), vous écrivez **la même table** dans les deux formats, vous la modifiez, et vous comparez ce que chacun range dans le stockage.

**Prérequis :** la partie 2 est terminée (la table `silver.orders` existe).

> **⚠️ Un seul notebook actif à la fois.** Arrêtez le noyau des notebooks précédents : panneau *Running Terminals and Kernels* > *Shut Down All*.

In [ ]:
import os
from collections import Counter

import boto3
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("tp1-iceberg-delta").getOrCreate()
s3 = boto3.client("s3", endpoint_url=os.environ["S3_ENDPOINT"])

ICEBERG = "lakehouse.gold.comparaison_iceberg"          # une table du catalogue, désignée par son nom
DELTA = "s3a://lakehouse/delta/comparaison_delta"        # une table Delta, désignée par son emplacement


def a_completer(quoi):
    """Signale clairement une partie du TP qui n'a pas encore été écrite."""
    raise NotImplementedError(f"⚠️ Cellule à compléter : {quoi}")


def fichiers(chemin):
    """Liste les fichiers d'un dossier du stockage S3, rangés par type."""
    bucket, prefixe = chemin.replace("s3a://", "").replace("s3://", "").split("/", 1)
    cles = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefixe + "/"):
        cles += [o["Key"][len(prefixe) + 1:] for o in page.get("Contents", [])]
    return cles


def resume(cles):
    types = Counter("données Parquet" if c.endswith(".parquet") else
                    "métadonnées " + c.rsplit(".", 1)[-1].upper() for c in cles if not c.endswith("/"))
    return dict(sorted(types.items()))


commandes = spark.table("lakehouse.silver.orders").select(
    "order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_delivered_customer_date")
print(commandes.count(), "commandes")

## Étape 1 — Écrire la même table dans les deux formats

La table Iceberg est **créée dans le catalogue** : on la désigne ensuite par son nom, et tous les outils la trouvent (Spark, Trino, Superset). La cellule est fournie.

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.gold")
commandes.writeTo(ICEBERG).using("iceberg").createOrReplace()

# Dossier de la table : celui qui contient son sous-dossier data/
emplacement_iceberg = spark.sql(f"SELECT file_path FROM {ICEBERG}.files LIMIT 1").first()[0].split("/data/")[0]
print("Iceberg :", emplacement_iceberg)

Écrivez maintenant la même table au format **Delta**, à l'emplacement `DELTA`, en remplaçant son contenu si elle existe : `commandes.write.format("delta").mode("overwrite").save(DELTA)`.

In [ ]:
a_completer("écriture de la table Delta")  # À COMPLÉTER
print("Delta   :", DELTA, "—", spark.read.format("delta").load(DELTA).count(), "lignes")

Regardez ce que chaque format a écrit dans le stockage.

In [ ]:
f_iceberg, f_delta = fichiers(emplacement_iceberg), fichiers(DELTA)
print("Iceberg :", resume(f_iceberg))
print("Delta   :", resume(f_delta))
print("\nMétadonnées Iceberg :", sorted(c for c in f_iceberg if c.startswith("metadata/")))
print("Journal Delta       :", sorted(c for c in f_delta if c.startswith("_delta_log/")))

> **❓ Question 1.** Qu'ont en commun les deux tables ? Où chaque format range-t-il ses métadonnées, et sous quelle forme ?

## Étape 2 — Modifier la table : une nouvelle version

Le service client corrige l'orthographe d'un statut : `canceled` devient `cancelled`. Même modification dans les deux tables.

In [ ]:
spark.sql(f"UPDATE {ICEBERG} SET order_status = 'cancelled' WHERE order_status = 'canceled'")

Faites la même modification sur la table Delta. Une table Delta désignée par son emplacement s'écrit, en SQL, ``delta.`<chemin>` `` : ``spark.sql(f"UPDATE delta.`{DELTA}` SET ... WHERE ...")``.

In [ ]:
a_completer("UPDATE de la table Delta")  # À COMPLÉTER

Chaque format garde l'historique de ses versions. Pour Iceberg, c'est la table de métadonnées `snapshots` ; pour Delta, la commande ``DESCRIBE HISTORY delta.`<chemin>` ``. Complétez la seconde requête (colonnes `version`, `timestamp`, `operation`).

In [ ]:
spark.sql(f"SELECT committed_at, snapshot_id, operation FROM {ICEBERG}.snapshots ORDER BY committed_at").show(truncate=False)
a_completer("historique de la table Delta")  # À COMPLÉTER

In [ ]:
avant_i, avant_d = resume(f_iceberg), resume(f_delta)
f_iceberg, f_delta = fichiers(emplacement_iceberg), fichiers(DELTA)
print("Iceberg : avant", avant_i, "\n          après", resume(f_iceberg))
print("Delta   : avant", avant_d, "\n          après", resume(f_delta))

## Étape 3 — Revenir dans le passé

Les deux formats permettent de relire une version antérieure (*time travel*) : Iceberg par numéro de *snapshot*, Delta par numéro de version.

In [ ]:
premier = spark.sql(f"SELECT snapshot_id FROM {ICEBERG}.snapshots ORDER BY committed_at").first()[0]
compter = "SELECT order_status, COUNT(*) AS nb FROM {} WHERE order_status LIKE 'cancel%' GROUP BY order_status"

print("Iceberg, version initiale :")
spark.sql(compter.format(f"{ICEBERG} VERSION AS OF {premier}")).show()
print("Delta, version 0 :")
spark.read.format("delta").option("versionAsOf", 0).load(DELTA).createOrReplaceTempView("delta_v0")
spark.sql(compter.format("delta_v0")).show()
print("Delta, version actuelle :")
spark.sql(compter.format(f"delta.`{DELTA}`")).show()

> **❓ Question 2.** Après l'`UPDATE`, combien de fichiers Parquet chaque table compte-t-elle ? Pourquoi les anciens fichiers sont-ils toujours là, et quand finissent-ils par disparaître ?

## Étape 4 — Qui peut lire la table ?

Le catalogue Iceberg connaît la table Iceberg : n'importe quel moteur branché sur ce catalogue la trouve par son nom. La table Delta, elle, n'existe que par son emplacement.

In [ ]:
spark.sql("SHOW TABLES IN lakehouse.gold").filter("tableName LIKE 'comparaison%'").show()

> **❓ Question 3.** Le TP 3 interroge les tables avec Trino et les affiche dans Superset, en passant par le catalogue. Qu'est-ce que cela implique pour la table Delta ? Pourquoi les deux formats existent-ils, et comment le marché évolue-t-il ?

## Pour finir

Supprimez les deux tables de démonstration.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {ICEBERG} PURGE")
for page in s3.get_paginator("list_objects_v2").paginate(Bucket="lakehouse", Prefix="delta/comparaison_delta/"):
    for o in page.get("Contents", []):
        s3.delete_object(Bucket="lakehouse", Key=o["Key"])
print("tables de démonstration supprimées")